# EXP-2026-004 / Q5-A — 환자 수준 S-beat 실패 지도 (quest50)

**상태: DESIGN / RESULT NOT RUN.** 이 notebook에는 아직 어떤 측정 결과도 없다.

- **ANALYSIS ONLY / NO TRAINING** — Q5-A는 학습하지 않는다. 저장된 예측(logits/probabilities)과 처리 완료 beat 배열만 읽는다.
- 이 실험은 **원인 확정 실험이 아니다.** 여기서 말할 수 있는 것은 `failure-associated factor`(**실패 연관 요인**)와 차기 개입 가설까지이며, 실제 `원인` 여부는 Q5-B에서 그 요인 하나만 바꾸는 개입과 음성대조군으로 검증한다.
- 근거가 불충분하면 `UNRESOLVED` / `INSUFFICIENT_ARTIFACTS` / `DATA_INTEGRITY_BLOCKED`를 정식 판정으로 기록한다. 억지로 다음 모델을 고르지 않는다.
- **residual CNN 경로는 closed**(Q4-O NO-GO, Q4-Q mechanism+utility fail)이며 이 분석에서 재개하거나 변형 residual architecture를 제안하지 않는다. **INCART rescue run도 하지 않는다.**

spec: `experiments/specs/EXP-2026-004-q5a-patient-failure-atlas.md`

## mode 실행 순서 (정확히 하나만 활성)

| 순서 | mode | 하는 일 |
|---|---|---|
| ① | — | 셀 2: repo 준비 + commit SHA 표시 |
| ② | — | 셀 2: Q4-O / Q4-P / Q4-Q / Q5-A 회귀 테스트 |
| ③ | — | 셀 3: Drive mount + 경로 |
| ④ | `INVENTORY` | 셀 4: 후보 run·prediction artifact 검색·검증 (분석 결과 없음) |
| ⑤ | `INVENTORY` | 셀 5: inventory gate 결과 확인 — 실패하면 여기서 중단 |
| ⑥ | `ANALYZE` | 셀 6: 모든 gate 통과 시에만 전체 분석 |
| ⑦ | `REPORT` | 셀 7: 저장 bundle만 다시 표시 (재계산 없음) |

기본값은 `DESIGN`(데이터 접근 없음). gate가 실패하면 조용한 fallback이나 위치 기반 억지 매칭을 하지 않고, 누락 파일과 해결 방법을 표로 보여준 뒤 중단한다.

## 입력 자산 (Drive) — 2026-08-09 INVENTORY 실측 반영

- **atlas source / frozen source**: `MyDrive/mitbih/mamba_data.npz` (file id `1p3HvC_bnbiQlEanFOVIvVdejy60W0tho`) — `beat/ref/feats/y/pid/t`. `t`가 annotation sample index라 안정 키의 근거가 된다
- **주석 캐시**: `MyDrive/mitbih/raw_ann/mitdb/` (folder id `151DJAcjCbDXCoy9ZIPudbtSuVziG1fnj`) — `.hea`+`.atr`. **`mitbih/mitdb/`가 아니다**(ASSETS.md 경로 정정). 원 symbol(A/a/J/S) 복구용
- **V10 (primary)**: `MyDrive/mitbih/ablation_step9d/pwave/ens.npz` — `colab_step9d_final.py :: run_final("pwave")`
- **BASE26 (paired control)**: `MyDrive/mitbih/ablation_step9d/base26/ens.npz` — 같은 스크립트·같은 seed, **P파 특징 블록만** 다름
- **V9 `kink_noctx`: ARTIFACT_ABSENT** — Drive 전량 조회·repo grep에서 발견되지 않았다. 기록된 0.597은 **검증 불가**로 남기고 재학습하지 않는다
- 교차 검증용: `ecg_multi.npz` (file id `1aSj_1jvS_W2iruVnORIG6DTVuHobzNzq`) · Q4-Q run folder id `1ZCAYZCl4T4eoZzdFfV_IzkB0Mgbcqlw4` · `MyDrive/MedKOS/ecg-model/registry.jsonl`

legacy 산출물(`ens.npz`)에는 annotation index가 없다. Q5-A는 동결 source와 `pid`·`y`를 **전량 대조해 행 대응을 검증**한 뒤에만 `t`로 키를 부여하며, 한 행이라도 어긋나면 중단한다(row order 매칭 금지). legacy 산출물은 DS2만 채점하므로 분석 cohort는 모든 모델이 공통으로 덮는 DS2 record로 제한되고 제외 목록이 남는다.

이 notebook의 첫 임무는 남아 있는 산출물로 V10의 지표를 **재계산해 확정**하고, 확정할 수 없는 것(V9 0.597)은 그 이유를 기록하는 것이다.


In [ ]:
# ── cell 1: mode config — 정확히 하나만 활성 ────────────────────────────
MODE = "DESIGN"   # "DESIGN" | "INVENTORY" | "ANALYZE" | "REPORT"

VALID_MODES = ("DESIGN", "INVENTORY", "ANALYZE", "REPORT")
assert MODE in VALID_MODES, f"MODE must be exactly one of {VALID_MODES}"
assert sum(MODE == m for m in VALID_MODES) == 1
print(f"MODE = {MODE}")
print("EXP-2026-004 / Q5-A · ANALYSIS ONLY / NO TRAINING")
if MODE == "DESIGN":
    print("DESIGN mode: 아무것도 실행하지 않는다. FULL RESULT NOT RUN.")

In [ ]:
# ── cell 2: repo 준비 + commit SHA + 회귀 테스트 ────────────────────────
import os, subprocess, sys
os.chdir("/content")
REPO = "/content/my-github-test"
URL = "https://github.com/ehdbddl06001-ui/my-github-test.git"
BRANCH = "main"   # 아직 병합 전이면 작업 브랜치명을 넣는다
RUN_TESTS = True

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["rm", "-rf", REPO], check=False)
    r = subprocess.run(["git", "clone", URL, REPO], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr); raise SystemExit("git clone failed")
for cmd in (["git", "-C", REPO, "fetch", "origin", BRANCH],
            ["git", "-C", REPO, "checkout", "-B", BRANCH,
             f"origin/{BRANCH}"]):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout, r.stderr)
        raise SystemExit(f"git failed: {' '.join(cmd)} - BRANCH 이름을 확인한다")
os.chdir(REPO)
print("branch:", subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip())
sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True)
print("repo commit SHA:", sha.stdout.strip())

for name in ("q5a_patient_failure_atlas",
             "q4q_transportability_replication",
             "q4p_best_epoch_zero_diagnostic",
             "q4o_leakage_free_residual"):
    sys.modules.pop(name, None)
sys.path = [p for p in sys.path if "mit-bih" not in p]
sys.path.insert(0, os.path.join(REPO, "mit-bih"))
import q5a_patient_failure_atlas as QA

NEED_Q5A = 2   # 이 notebook이 요구하는 최소 모듈 버전
assert QA.MODULE_VERSION >= NEED_Q5A, (
    f"stale module: q5a v{QA.MODULE_VERSION} < v{NEED_Q5A}. BRANCH가 맞는지 "
    "확인하고 Restart runtime 후 이 셀을 다시 실행한다")
if RUN_TESTS:
    for suite in ("test_q4o_leakage_free_residual",
                  "test_q4p_best_epoch_zero_diagnostic",
                  "test_q4q_transportability_replication",
                  "test_q5a_patient_failure_atlas"):
        rr = subprocess.run([sys.executable, f"mit-bih/{suite}.py"],
                            capture_output=True, text=True)
        tail = (rr.stdout + "\n" + rr.stderr).splitlines()[-3:]
        print(f"[{suite}] rc={rr.returncode} | " + " | ".join(t.strip() for t in tail))
        if rr.returncode != 0 and suite.endswith("q5a_patient_failure_atlas"):
            print("\n".join((rr.stdout + "\n" + rr.stderr).splitlines()[-30:]))
            raise SystemExit("Q5-A test suite failed - fix before running")
print("module ok:", QA.self_check()["module_build"])
print("analysis only:", QA.self_check()["analysis_only"], "| status:", QA.STATUS)

In [ ]:
# ── cell 3: Drive mount + 경로 (DESIGN이면 skip) ────────────────────────
if MODE == "DESIGN":
    print("skip - DESIGN mode. RESULT NOT RUN.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    ECG_ROOT = f"{DRIVE}/MedKOS/ecg-model"
    RUNS_ROOT = f"{ECG_ROOT}/runs"
    REGISTRY = f"{ECG_ROOT}/registry.jsonl"
    MITBIH = f"{DRIVE}/mitbih"
    # frozen source = atlas source: mamba_data.npz carries beat/y/pid AND the
    # annotation sample index `t` that every derived key rests on.
    ATLAS_SOURCE = f"{MITBIH}/mamba_data.npz"   # id 1p3HvC_bnbiQlEanFOVIvVdejy60W0tho
    DATA_MULTI   = f"{MITBIH}/ecg_multi.npz"    # id 1aSj_1jvS_W2iruVnORIG6DTVuHobzNzq
    ANN_DIR = QA.find_annotation_dir(MITBIH)    # measured: raw_ann/mitdb
    import time as _t
    TS = _t.strftime("%Y%m%dT%H%M")
    OUT_INV = f"{RUNS_ROOT}/{TS}_EXP-2026-004_q5a_inventory"
    OUT_RUN = f"{RUNS_ROOT}/{QA.run_dir_name(TS)}"
    print("atlas/frozen source :", ATLAS_SOURCE, os.path.exists(ATLAS_SOURCE))
    print("annotation dir      :", ANN_DIR, "(None이면 subtype 블록 unavailable)")
    print("runs root           :", RUNS_ROOT, os.path.isdir(RUNS_ROOT))
    print("ablation root       :", MITBIH, os.path.isdir(MITBIH))
    print("registry            :", REGISTRY, os.path.exists(REGISTRY))
    print("inventory out       :", OUT_INV)
    print("analysis out        :", OUT_RUN)
    print("출력은 항상 새 versioned 경로 - 기존 bundle은 절대 덮어쓰지 않는다.")
    if MODE in ("INVENTORY", "ANALYZE") and not os.path.exists(ATLAS_SOURCE):
        raise SystemExit(f"missing input: {ATLAS_SOURCE} - fix the path, do not guess")


In [ ]:
# ── cell 4: INVENTORY — 후보 run·prediction artifact 검색/검증 ──────────
if MODE != "INVENTORY":
    print(f"skip - MODE={MODE}")
else:
    import json
    os.makedirs(OUT_INV, exist_ok=True)
    log = QA.Q4O.RunLog()
    INV = QA.scan_inventory([RUNS_ROOT, MITBIH],
                            REGISTRY if os.path.exists(REGISTRY) else None,
                            log=log)
    FREEZE = QA.freeze_baseline(INV)
    QA._dump_json(f"{OUT_INV}/source_inventory.json", INV)
    QA._dump_csv(f"{OUT_INV}/source_inventory.csv",
                 [{k: v for k, v in e.items() if k != "prediction_files"}
                  for e in INV["entries"]] or [{"run_id": ""}])
    QA._dump_json(f"{OUT_INV}/baseline_freeze.json", FREEZE)
    print(f"\ncandidates: {INV['n_candidates']}")
    for e in INV["entries"]:
        print(f"  {e['run_id'][:46]:46s} model={str(e['model_name'])[:20]:20s} "
              f"beat-level={'Y' if e['beat_level_ready'] else 'N'} "
              f"metric={str(e['metric_definition'])[:18]}")
    print("\nbaseline freeze status:", FREEZE["status"])
    for r in FREEZE["reasons"]:
        print("  reason:", r)
    print("beat-level models :", FREEZE["beat_level_models"])
    print("aggregate-only    :", FREEZE["aggregate_only_models"])

In [ ]:
# ── cell 5: inventory gate 결과 확인 (실패하면 여기서 중단) ─────────────
if MODE != "INVENTORY":
    print(f"skip - MODE={MODE}")
else:
    GATES = QA.evaluate_artifact_gates(INV, FREEZE, None)
    QA._figure_gate_dashboard(OUT_INV, INV, FREEZE, GATES)
    from IPython.display import Image, display
    display(Image(f"{OUT_INV}/figures/inventory_gate_dashboard.png"))
    print("gate pass:", GATES["pass"])
    if not GATES["pass"]:
        print("\n누락/모호성 표 - 해결 전에는 ANALYZE로 넘어가지 않는다:")
        for s in GATES["stops"]:
            print("  STOP:", s)
        print("\n다음 일은 모델 실험이 아니라 artifact 복구/adapter 수정이다.")
        print("그래도 ANALYZE를 실행하면 BLOCKED_MEASURED bundle이 저장된다"
              " (숨기지 않는 정식 결과).")
    else:
        print("gate 통과 - 셀 1에서 MODE를 'ANALYZE'로 바꾸고 셀 3, 6을 실행한다.")

In [ ]:
# ── cell 6: ANALYZE — 전체 실패 지도 (gate 통과 시에만) ─────────────────
if MODE != "ANALYZE":
    print(f"skip - MODE={MODE}")
else:
    import glob, json
    inv_dirs = sorted(glob.glob(f"{RUNS_ROOT}/*_EXP-2026-004_q5a_inventory"))
    if not inv_dirs:
        raise SystemExit("INVENTORY를 먼저 실행한다 (source_inventory.json 없음)")
    INV_DIR = inv_dirs[-1]
    print("using inventory:", INV_DIR)
    argv = ["--mode", "ANALYZE", "--data", ATLAS_SOURCE,
            "--source", ATLAS_SOURCE,        # 행 대응 검증의 기준
            "--inventory", INV_DIR, "--out", OUT_RUN]
    if ANN_DIR:
        argv += ["--ann-dir", ANN_DIR]       # 원 symbol -> subtype 블록
    rc = QA.main(argv)
    res = json.load(open(f"{OUT_RUN}/result.json", encoding="utf-8"))
    print("\nstatus:", res["status"], "| training_performed:", res["training_performed"])
    print("branch:", res["decision"]["branch"], f"({res['decision']['rule']})")
    print("reason:", res["decision"]["reason"])
    print("analysis records:", res["split"].get("ds2_analysis"))
    print("excluded (not covered by every model):", res["split"].get("ds2_excluded"))
    for a in res["baseline_freeze"].get("absent_baselines", []):
        print("absent baseline:", a["label"], "|", a["status"], "|", a["consequence"])


In [ ]:
# ── cell 7: REPORT — 저장 bundle만 다시 표시 (재계산 없음) ──────────────
if MODE != "REPORT":
    print(f"skip - MODE={MODE}")
else:
    import glob
    dirs = sorted(glob.glob(f"{RUNS_ROOT}/*_EXP-2026-004_q5a_patient_failure_atlas"))
    if not dirs:
        print("RESULT NOT RUN - 저장된 Q5-A bundle이 없다.")
    else:
        RUN_DIR = dirs[-1]
        rep = QA.report_bundle(RUN_DIR)
        print("run   :", RUN_DIR)
        print("status:", rep["status"], "| recomputed:", rep["recomputed"])
        print("branch:", (rep["decision"] or {}).get("branch"))
        for lab, m in rep["model_metrics"].items():
            ps = m["patient_summary"]
            print(f"  {lab}: beat-micro {m['beat_micro_s_prauc']:.4f} | "
                  f"record-macro {m['record_macro_s_prauc']:.4f} | "
                  f"p10 {ps['p10']:.4f} | worst "
                  + ", ".join(f"{w['record']}:{w['value']:.3f}" for w in ps['worst5']))
        from IPython.display import Image, display, Markdown
        for f in rep["figures"]:
            display(Image(f"{RUN_DIR}/figures/{f}"))
        display(Markdown(rep["summary_md"] or "_no summary_"))

In [ ]:
# ── cell 8: 한국어 자동 요약 (bundle 없으면 RESULT NOT RUN) ─────────────
import glob as _g, os as _o
_dirs = _g.glob("/content/drive/MyDrive/MedKOS/ecg-model/runs/"
                "*_EXP-2026-004_q5a_patient_failure_atlas")
if not _dirs or not _o.path.exists(f"{sorted(_dirs)[-1]}/summary.md"):
    print("RESULT NOT RUN - 이 세션에는 측정된 Q5-A bundle이 없다.")
    print("확인된 것: 없음 / 확인되지 않은 것: 전부")
    print("이 실험은 '원인'이 아니라 '실패 연관 요인'을 찾는 관찰적 분석이다.")
    print("다음 실험(Q5-B)은 아직 실행하지도, 구현하지도 않았다.")
else:
    from IPython.display import Markdown, display
    display(Markdown(open(f"{sorted(_dirs)[-1]}/summary.md", encoding="utf-8").read()))

## 해석 경계 (사전 등록 요약)

- Q5-A가 내놓는 것은 **실패 연관 요인**과 차기 개입 가설이다. 관찰적 사후 분석이므로 `원인`을 확정하지 않는다.
- DS2는 이 프로젝트에서 반복 사용된 cohort다 — `untouched external test`가 아니라 **descriptive failure audit**이다.
- threshold·bin·proxy 정의·branch rule은 DS1(또는 사전 지정 값)에서만 정하고, DS2 label을 보고 바꾸지 않는다.
- CI는 환자(record) 단위 bootstrap이다. beat bootstrap으로 환자 수 부족을 가리지 않는다.
- P-wave annotation ground truth는 존재하지 않는다. 여기의 모든 atrial 지표는 **proxy**이며, V10이 이미 쓰는 feature와 겹치는 proxy는 그렇게 표시된다.
- 근거가 갈리면 `UNRESOLVED`, artifact가 부족하면 `INSUFFICIENT_ARTIFACTS` / `BLOCKED_MEASURED`로 기록한다. 이것도 정식 결과다.
- **Q5-B는 이 PR에서 구현하지 않는다.** Q5-A가 측정되고 사용자가 분기를 승인한 뒤 별도 PR에서 만든다.